# Load the data

In [14]:
import pandas as pd

df = pd.read_parquet("combined.parquet")
print(df.shape)
df.head()


(6940494, 31)


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN_CITY_NAME,DEST_CITY_NAME,CRS_DEP_TIME,...,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,ARR_DELAY,DIVERTED,AIR_TIME
0,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"New York, NY","Los Angeles, CA",659,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Los Angeles, CA","New York, NY",2200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Madison, WI","Charlotte, NC",644,...,20.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Philadelphia, PA","Las Vegas, NV",1855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,"Chicago, IL","Phoenix, AZ",830,...,0.0,0.0,44.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [15]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df = df.drop_duplicates()
print(df.shape)


C:\Users\denni\AppData\Local\Temp\ipykernel_38888\2560387958.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])


(6400744, 31)


# Clean rows

Remove cancelled flights and rows where `DEP_DELAY` (our target source) is missing.

In [16]:
df = df[df["CANCELLED"] == 0].copy()
print("After dropping cancellations:", df.shape)


After dropping cancellations: (6304794, 31)


In [17]:
df = df.dropna(subset=["DEP_DELAY"])
df["DEP_DELAY"] = df["DEP_DELAY"].astype(int)
print("After dropping missing target:", df.shape)

After dropping missing target: (6304794, 31)


# Create multiclass target

Bin `DEP_DELAY` into 5 classes: no delay, 15–29 min, 30–59 min, 1–2 hours, 2+ hours.

In [18]:
def categorize_delay(minutes):
    if minutes < 15:    return 0  # No delay
    elif minutes < 30:  return 1  # 15–29 min
    elif minutes < 60:  return 2  # 30–59 min
    elif minutes < 120: return 3  # 1–2 hours
    else:               return 4  # 2+ hours

df["DELAY_CLASS"] = df["DEP_DELAY"].apply(categorize_delay)
df["DELAY_CLASS"].value_counts(normalize=True).sort_index()

DELAY_CLASS
0    0.780265
1    0.072238
2    0.066119
3    0.048023
4    0.033354
Name: proportion, dtype: float64

# Feature engineering

Split `"City, ST"` into separate city/state columns, and extract `DEP_HOUR` (0–23) from the HHMM `CRS_DEP_TIME`.

In [19]:
df[["ORIGIN_CITY", "ORIGIN_STATE"]] = df["ORIGIN_CITY_NAME"].str.split(",", expand=True)
df[["DEST_CITY", "DEST_STATE"]] = df["DEST_CITY_NAME"].str.split(",", expand=True)

df["ORIGIN_STATE"] = df["ORIGIN_STATE"].str.strip()
df["DEST_STATE"] = df["DEST_STATE"].str.strip()

df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100

# Drop columns we don't need

Drop columns only known after the flight, redundant/constant columns, and high-cardinality city columns (keeping state instead).

In [20]:
drop_cols = [
    # only known after the flight lands
    "DEP_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "ARR_TIME",
    "ARR_DEL15",
    "AIR_TIME",
    "CARRIER_DELAY",
    # already filtered or not useful
    "CANCELLED",
    "CANCELLATION_CODE",
    "DEST_CITY_MARKET_ID",
    # redundant
    "FL_DATE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    # replaced by state. Too many columns for one-hot
    "ORIGIN_CITY_NAME",
    "DEST_CITY_NAME",
    "ORIGIN_CITY",
    "DEST_CITY",
    "DEP_DELAY",
    #All data is from the year of 2025
    "YEAR",
    "ARR_DELAY",
    "DEP_DEL15",
    "DIVERTED",
    "DEST_AIRPORT_SEQ_ID"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)
df.head()


(6304794, 14)


,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,DISTANCE,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DELAY_CLASS,ORIGIN_STATE,DEST_STATE,DEP_HOUR
0,1,1,1,3,AA,2475.0,NaN,NaN,NaN,NaN,0,NY,CA,6
1,1,1,1,3,AA,2475.0,NaN,NaN,NaN,NaN,0,CA,NY,22
2,1,1,1,3,AA,708.0,0.0,0.0,0.0,0.0,2,WI,NC,6
3,1,1,1,3,AA,2176.0,NaN,NaN,NaN,NaN,0,PA,NV,18
4,1,1,1,3,AA,1440.0,0.0,44.0,0.0,0.0,0,IL,AZ,8


# Save preprocessed data

In [ ]:
df.to_parquet('combined_preprocessed.parquet', index=False, compression='zstd')